# 20 — Resultados adicionais e figuras

Desenvolve `app/graficos.py` (as seis figuras dos resultados). **F11, F16.**

In [2]:
import sys, os, tempfile
_cwd = os.getcwd()
RAIZ = os.path.dirname(_cwd) if os.path.basename(_cwd) == 'tests' else _cwd
if RAIZ not in sys.path:
    sys.path.insert(0, RAIZ)
import matplotlib
matplotlib.use('Agg')                      # sem display: so escreve arquivo
import matplotlib.pyplot as plt
import numpy as np, pandas as pd
from app import nucleo
from app.principal import executar_pipeline
from app.mercado import RendaVariavel

## Desenvolvimento

As funções abaixo foram escritas aqui e, após os testes, movida(s) para `app/graficos.py`.

In [3]:
DESTINO_PADRAO = "results"

# Grade de γ do gráfico de sensibilidade. Exige uma reotimização por ponto, que
# é o trecho caro da geração (~12 s com 4 milhões de cenários).
GRADE_GAMMA = (1.5, 2.0, 3.0, 5.0, 8.0, 10.0, 15.0, 20.0)

In [4]:
def _rodape(fig, texto: str) -> None:
    """Linha de procedência: subordinada à legenda do documento, não a substitui."""
    fig.text(0.5, 0.012, texto, ha="center", fontsize=6.5, color="0.45")
    fig.subplots_adjust(bottom=0.22)

In [5]:
def _salvar(fig, destino: str, nome: str, rodape: str) -> str:
    _rodape(fig, rodape)
    caminho = os.path.join(destino, nome)
    fig.savefig(caminho, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return caminho

In [6]:
def montar_rodape(res: dict, cfg: dict, periodo: tuple[str, str], n_obs: int,
                  beta_anual: float, anos: float, unidade: str) -> str:
    """Texto de procedência impresso em todas as figuras."""
    return (f"Ibovespa {unidade} · {periodo[0]} a {periodo[1]} ({n_obs} obs) · "
            f"γ={cfg['gamma']:g} · β={beta_anual:g} a.a. · T={anos:g} anos · "
            f"{cfg['n_scenarios']:,} cenários · seed {cfg['seed']} · "
            f"α*={res['alpha_star'][0]:.4f}".replace(",", "."))

In [7]:
def gerar(res: dict, mercado, rf: float, cfg: dict, rodape: str,
          periodos_por_ano: int, destino: str = DESTINO_PADRAO) -> list[str]:
    """Gera as seis figuras e devolve os caminhos escritos.

    ``res`` é a saída de ``executar_pipeline``; ``mercado`` e ``rf`` são
    necessários para os dois gráficos que reotimizam (G(α) e α* × γ).
    """
    os.makedirs(destino, exist_ok=True)
    g = float(cfg["gamma"])
    Rf = 1.0 + rf
    T = res["horizonte"]
    escritos = []

    # Cenários reaproveitados pelos dois gráficos que reotimizam — amostrar uma
    # vez só mantém as curvas coerentes com o α* reportado.
    R = np.maximum(1.0 + mercado.amostrar(cfg["n_scenarios"], seed=cfg["seed"]), 0.0)

    # ── 1. G(α) × α, marcando a raiz ────────────────────────────────────────
    a_star = float(res["alpha_star"][0])
    grade = np.linspace(a_star - 1.0, a_star + 1.0, 60)
    G = [nucleo.funcao_foc(np.array([a]), R, Rf, g)[0] for a in grade]
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.axhline(0, color="0.7", lw=0.8)
    ax.plot(grade, G, color="#1f77b4")
    ax.plot([a_star], [0], "o", color="#d62728", zorder=5,
            label=f"α* = {a_star:.4f}")
    ax.set_xlabel("α"); ax.set_ylabel("G(α)")
    ax.set_title("Condição de primeira ordem: G(α) = 0")
    ax.legend()
    escritos.append(_salvar(fig, destino, "foc_G_de_alpha.png", rodape))

    # ── 2. α* × γ (hipérbole de Merton) ─────────────────────────────────────
    alphas = [nucleo.resolver_alpha_otimo(R, Rf, gi)[0] for gi in GRADE_GAMMA]
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(GRADE_GAMMA, alphas, "o-", color="#1f77b4")
    ax.axvline(g, color="0.7", ls="--", lw=0.8)
    ax.set_xlabel("γ (aversão relativa ao risco)"); ax.set_ylabel("α*")
    ax.set_title("Sensibilidade da carteira ótima à aversão ao risco")
    escritos.append(_salvar(fig, destino, "alpha_vs_gamma.png", rodape))

    # ── 3. α* × T — a miopia (F12) ──────────────────────────────────────────
    # α* não depende de T (a CPO da carteira é atemporal), então não há
    # reotimização: a linha horizontal É o resultado.
    Ts = np.array([1, T // 4, T // 2, 3 * T // 4, T])
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(Ts, np.full(Ts.shape, a_star), "o-", color="#2ca02c")
    ax.set_ylim(a_star - 0.05, a_star + 0.05)
    ax.set_xlabel("T (períodos restantes)"); ax.set_ylabel("α*")
    ax.set_title("Miopia: α* invariante ao horizonte")
    escritos.append(_salvar(fig, destino, "miopia_alpha_vs_T.png", rodape))

    # ── 4. θ_t ao longo do tempo (F13) ──────────────────────────────────────
    theta = res["theta"]
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(np.arange(len(theta)), theta, color="#1f77b4")
    ax.set_yscale("log")          # sem log, θ_t fica rente a zero e só dispara no fim
    ax.set_xlabel("t (períodos)"); ax.set_ylabel(r"$\theta_t$ (escala log)")
    ax.set_title(r"Fração de consumo $\theta_t$ — crescente até $\theta_T = 1$")
    escritos.append(_salvar(fig, destino, "theta_t.png", rodape))

    # ── 5. W_t com banda P5–P95 ─────────────────────────────────────────────
    t = np.arange(T + 1)
    media, p5, p95 = (res["trajetoria_W_media"], res["trajetoria_W_p5"],
                      res["trajetoria_W_p95"])
    fig, (ax, ax2) = plt.subplots(2, 1, figsize=(7, 5.4), sharex=True,
                                  gridspec_kw={"height_ratios": [3, 1]})
    ax.fill_between(t, p5, p95, color="#1f77b4", alpha=0.3, label="P5–P95")
    ax.plot(t, media, color="#1f77b4", label="média")
    ax.set_yscale("log")
    ax.set_ylabel(r"$W_t$ (escala log)")
    ax.set_title("Trajetória da riqueza")
    ax.legend()
    # A banda acima é real mas estreita demais para ser vista numa escala que
    # cobre três ordens de grandeza; aqui ela aparece em termos relativos.
    largura = 100.0 * (p95 - p5) / np.where(media > 0, media, np.nan)
    ax2.plot(t, largura, color="#7f7f7f")
    ax2.set_xlabel("t (períodos)")
    ax2.set_ylabel("P95-P5\n(% da média)", fontsize=8)
    escritos.append(_salvar(fig, destino, "riqueza_W_t.png", rodape))

    # ── 6. Consumo agregado por ano ─────────────────────────────────────────
    c = res["trajetoria_c_media"]
    n_anos = max(1, T // periodos_por_ano)
    por_ano = [c[i * periodos_por_ano:(i + 1) * periodos_por_ano].sum()
               for i in range(n_anos)]
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(np.arange(1, n_anos + 1), por_ano, color="#ff7f0e")
    ax.set_xlabel("ano"); ax.set_ylabel(r"consumo (fração de $W_0$)")
    ax.set_title("Consumo agregado por ano")
    escritos.append(_salvar(fig, destino, "consumo_por_ano.png", rodape))

    return escritos

**Teste** — resultado base sobre o qual as figuras são desenhadas.

In [8]:
ret = pd.DataFrame({'data': pd.bdate_range('2022-05-24', periods=400).strftime('%Y-%m-%d'),
                    'ibov': 0.0007 + np.random.default_rng(0).normal(0, 0.011, 400),
                    'cdi': np.full(400, 0.00049)})
res = executar_pipeline({'retornos': ret, 'ativos': ['ibov'], 'periodos_por_ano': 252,
                         'gamma': 5.0, 'beta_anual': 0.96, 'w0': 1.0, 'horizonte': 252,
                         'n_scenarios': 200_000, 'n_paths': 500, 'seed': 1})
mkt = RendaVariavel(ret[['data', 'ibov']])
T = res['horizonte']
print('alpha* =', res['alpha_star'][0], '| T =', T)

alpha* = -0.36744844636966967 | T = 252


**Teste** — Etapas 3 e 7: `A_t` e a função valor `V_t` (F11).

In [9]:
# Etapas 3 e 7 do paper: A_t e V_t agora saem do pipeline (F11).
print('A_0 =', res['A_t'][0], '| A_T =', res['A_t'][-1], '| V_T =', res['valor_V'][-1])

A_0 = 779604271647.1617 | A_T = 1.0 | V_T = -0.25


In [10]:
assert res['A_t'].shape == (T + 1,) and res['valor_V'].shape == (T + 1,)
assert np.isclose(res['A_t'][-1], 1.0)                        # A_T = 1 (cond. terminal)
assert np.allclose(res['valor_V'], nucleo.funcao_valor(res['A_t'], 1.0, 5.0))
assert np.isclose(res['valor_V'][-1], 1.0 / (1 - 5.0))        # V_T(1) = 1/(1-g)
assert np.allclose(res['theta'], res['A_t'] ** (-1 / 5.0))    # theta_t = A_t^(-1/g)
print('F11 A_t / valor_V: PASSOU')

F11 A_t / valor_V: PASSOU


**Teste** — percentis por período, que sustentam a banda de confiança.

In [11]:
# Percentis periodo a periodo: sem eles nao ha banda de confianca.
p5, p50, p95 = res['trajetoria_W_p5'], res['trajetoria_W_mediana'], res['trajetoria_W_p95']
print('em t=T:  P5=%.6f  mediana=%.6f  P95=%.6f' % (p5[-1], p50[-1], p95[-1]))
for k in ('trajetoria_W_p5', 'trajetoria_W_mediana', 'trajetoria_W_p95'):
    assert res[k].shape == (T + 1,), k
assert np.all(p5 <= p50) and np.all(p50 <= p95)               # ordenacao
assert np.allclose([p5[0], p50[0], p95[0]], 1.0)              # todos partem de W_0
assert p95[-1] - p5[-1] > 0                                    # a banda tem largura em T
print('percentis por periodo: PASSOU')

em t=T:  P5=0.003878  mediana=0.004294  P95=0.004742
percentis por periodo: PASSOU


**Teste** — o rodapé de procedência impresso em cada figura.

In [12]:
# O rodape carrega a procedencia: sem ele, um PNG solto nao diz com que
# parametros foi gerado. Optou-se por imprimir na figura em vez de um JSON ao
# lado — a legenda e' impossivel de separar do resultado.
cfg = {'gamma': 5.0, 'n_scenarios': 50_000, 'seed': 1}
rodape = montar_rodape(res, cfg, ('2022-05-24', '2024-01-01'), len(ret), 0.96, 1.0, 'diario')
print(rodape)

Ibovespa diario · 2022-05-24 a 2024-01-01 (400 obs) · γ=5 · β=0.96 a.a. · T=1 anos · 50.000 cenários · seed 1 · α*=-0.3674


In [13]:
assert 'γ=5' in rodape and 'seed 1' in rodape and 'α*=' in rodape and '2022-05-24' in rodape
print('rodape de procedencia: PASSOU')

rodape de procedencia: PASSOU


**Teste** — geração das seis figuras.

In [14]:
# Gera num diretorio temporario para nao tocar em results/.
destino = os.path.join(tempfile.gettempdir(), 'graficos_dev20')
escritos = gerar(res, mkt, res['rf'], cfg, rodape, 252, destino=destino)
print('figuras:', [os.path.basename(c) for c in escritos])

figuras: ['foc_G_de_alpha.png', 'alpha_vs_gamma.png', 'miopia_alpha_vs_T.png', 'theta_t.png', 'riqueza_W_t.png', 'consumo_por_ano.png']


In [15]:
assert len(escritos) == 6
for c in escritos:
    assert os.path.exists(c) and os.path.getsize(c) > 5_000   # PNG nao-vazio
for c in escritos:
    os.remove(c)
os.rmdir(destino)
print('F16 graficos: PASSOU')

F16 graficos: PASSOU


**Teste** — `app.principal` não pode arrastar o matplotlib (impacto na web).

In [16]:
# O matplotlib NAO pode entrar pela esteira: a camada web importa app.principal
# e desenha no navegador. Se principal puxasse matplotlib, o servidor pagaria o
# custo de inicializacao e o risco de um backend nao thread-safe.
import subprocess
saida = subprocess.run(
    [sys.executable, '-c', "import sys; import app.principal; print('matplotlib' in sys.modules)"],
    capture_output=True, text=True, cwd=RAIZ).stdout.strip()
print('app.principal carrega matplotlib?', saida)

app.principal carrega matplotlib? False


In [17]:
assert saida == 'False'
print('isolamento do matplotlib: PASSOU')

isolamento do matplotlib: PASSOU
